# 14 - Model Comparison and Statistical Testing

In this notebook, we answer the question: Is Model B *actually* better than Model A, or did it just get lucky?

## Concept Overview
To mathematically prove one model is better, we run Cross-Validation to get an array of scores, and then use a Paired T-Test or Wilcoxon test to check for statistical significance.

## Visualizing Distributions
Let's train a Decision Tree and a Random Forest, get 10 scores for each via CV, and plot them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Get 10 scores for Model A
model_A = DecisionTreeClassifier(random_state=42)
scores_A = cross_val_score(model_A, X, y, cv=cv, scoring='accuracy')

# Get 10 scores for Model B
model_B = RandomForestClassifier(random_state=42)
scores_B = cross_val_score(model_B, X, y, cv=cv, scoring='accuracy')

plt.figure(figsize=(8, 6))
sns.boxplot(data=[scores_A, scores_B])
plt.xticks([0, 1], ['Decision Tree (Model A)', 'Random Forest (Model B)'])
plt.ylabel('Accuracy')
plt.title('10-Fold CV Score Distributions')
plt.show()

## Statistical Testing (Paired T-Test)
Looking at the boxplot, Model B seems better. But is it statistically significant? Let's run a Paired T-Test.

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

print(f"Model A Mean: {scores_A.mean():.4f}")
print(f"Model B Mean: {scores_B.mean():.4f}")
print("---")

# Paired T-Test
t_stat, p_value_t = ttest_rel(scores_A, scores_B)
print(f"Paired T-Test P-Value: {p_value_t:.4f}")

if p_value_t < 0.05:
    print("T-Test: Reject Null Hypothesis. The models are significantly different.")
else:
    print("T-Test: Accept Null Hypothesis. The difference is just noise.")
    
print("---")

# Wilcoxon Signed-Rank Test (Non-parametric, safer for CV scores)
w_stat, p_value_w = wilcoxon(scores_A, scores_B)
print(f"Wilcoxon P-Value: {p_value_w:.4f}")
if p_value_w < 0.05:
    print("Wilcoxon: Reject Null Hypothesis. The models are significantly different.")
else:
    print("Wilcoxon: Accept Null Hypothesis. The difference is just noise.")

Both tests yield a p-value < 0.05, mathematically proving that the Random Forest is superior.

## McNemar's Test
If you cannot use Cross-Validation (e.g., Deep Learning models that take 3 weeks to train), you only have ONE test set. In this case, use McNemar's Test on the actual predictions.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Train once
model_A.fit(X_tr, y_tr)
model_B.fit(X_tr, y_tr)

preds_A = model_A.predict(X_te)
preds_B = model_B.predict(X_te)

# Create contingency table
both_right = sum((preds_A == y_te) & (preds_B == y_te))
both_wrong = sum((preds_A != y_te) & (preds_B != y_te))
A_right_B_wrong = sum((preds_A == y_te) & (preds_B != y_te))
A_wrong_B_right = sum((preds_A != y_te) & (preds_B == y_te))

table = [[both_right, A_right_B_wrong],
         [A_wrong_B_right, both_wrong]]

print("Contingency Table:\n", np.array(table))

result = mcnemar(table, exact=True)
print(f"\nMcNemar P-Value: {result.pvalue:.4f}")

## Industry Notes
In high-stakes environments (finance, medicine), you must prove statistical significance before deploying a new model to replace an old one. A 0.5% boost might just be variance.

## Exercises
1. Modify `model_B` to be a `DecisionTreeClassifier(max_depth=5)`. Compare it to `model_A` using the T-Test. Is the difference significant?
2. Research why the Wilcoxon test is often preferred over the T-Test for cross-validation scores.